**Executed by Lakeflow Job run 886593886805341 · task `uc_tags_lineage_audit` · task run 475611867566209 · result SUCCESS · 2026-11-10 09:04:37 UTC → 2026-11-10 09:05:24 UTC · source `src/notebooks/02_governance_catalog.py`**

_Executed notebook exported from the job run; outputs are real._

# 02 · Unity Catalog governance – tags, ownership metadata, masks proof, lineage

Runs after the Lakeflow pipeline has published bronze/silver/gold. It:
1. Applies **governed tags** (workspace tag policies: `data_classification`, `class.*` PII classes, `system.certification_status`) plus project tags at table and column level.
2. **Proves the column masks work** – queries `silver_customers` as the job identity (not a PII reader).
3. Prints the **automatically captured lineage** (system.access.table_lineage) for the gold feature table.
4. Lists effective grants so an auditor can see who can read what.

In [2]:
dbutils.widgets.text("catalog", "serverless_stable_my49a8_catalog")
dbutils.widgets.text("fraud_schema", "northbridge_fraud")
CATALOG, FRAUD = dbutils.widgets.get("catalog"), dbutils.widgets.get("fraud_schema")
fq = lambda t: f"{CATALOG}.{FRAUD}.{t}"

In [3]:
# Governed tag keys defined by the workspace tag policies are reused (data_classification, class.*, system.certification_status);
# project-specific keys are prefixed nb_.
TABLE_TAGS = {
    "bronze_customers":      {"data_classification": "pii", "nb_access_tier": "restricted", "nb_regulation": "UK_GDPR"},
    "bronze_accounts":       {"data_classification": "pii", "nb_access_tier": "restricted", "nb_regulation": "UK_GDPR"},
    "bronze_payments":       {"nb_access_tier": "confidential", "nb_regulation": "PSR_APP_REIMBURSEMENT"},
    "bronze_payees":         {"nb_access_tier": "confidential"},
    "bronze_scam_claims":    {"nb_access_tier": "confidential", "nb_regulation": "PSR_APP_REIMBURSEMENT"},
    "silver_customers":      {"data_classification": "pii", "nb_access_tier": "confidential", "nb_pii_protection": "column_masked", "nb_regulation": "UK_GDPR"},
    "silver_accounts":       {"data_classification": "pii", "nb_access_tier": "confidential", "nb_pii_protection": "column_masked"},
    "silver_payments":       {"nb_access_tier": "confidential", "nb_regulation": "PSR_APP_REIMBURSEMENT"},
    "silver_scam_claims":    {"nb_access_tier": "confidential", "nb_regulation": "PSR_APP_REIMBURSEMENT"},
    "gold_payment_features": {"nb_access_tier": "internal", "nb_pii_protection": "no_direct_identifiers", "nb_ml_use": "training_and_scoring"},
    "gold_daily_fraud_kpis": {"nb_access_tier": "internal", "system.certification_status": "certified"},
    "gold_mule_watchlist":   {"nb_access_tier": "confidential", "nb_sharing": "interbank_data_sharing"},
}
PII_CLASS = {"full_name": "class.name", "date_of_birth": "class.date_of_birth", "email": "class.email_address",
             "mobile_phone": "class.phone_number", "postcode": "class.location", "sort_code": "class.bank_number", "account_number": "class.bank_number"}
COLUMN_TAGS = {
    "silver_customers": ["full_name", "date_of_birth", "email", "mobile_phone", "postcode"],
    "bronze_customers": ["full_name", "date_of_birth", "email", "mobile_phone", "postcode"],
    "silver_accounts":  ["sort_code", "account_number"],
    "bronze_accounts":  ["sort_code", "account_number"],
}
for t, tags in TABLE_TAGS.items():
    kv = ", ".join(f"'{k}' = '{v}'" for k, v in tags.items())
    spark.sql(f"ALTER TABLE {fq(t)} SET TAGS ({kv})")
for t, cols in COLUMN_TAGS.items():
    for c in cols:
        spark.sql(f"ALTER TABLE {fq(t)} ALTER COLUMN {c} SET TAGS ('{PII_CLASS[c]}')")
print("tags applied")

tags applied


### Tags as recorded in Unity Catalog

In [5]:
display(spark.sql(f"""
SELECT table_name, collect_list(concat(tag_name, '=', tag_value)) AS tags
FROM {CATALOG}.information_schema.table_tags WHERE schema_name = '{FRAUD}'
GROUP BY table_name ORDER BY table_name"""))
display(spark.sql(f"""
SELECT table_name, column_name, tag_name AS governed_pii_class
FROM {CATALOG}.information_schema.column_tags WHERE schema_name = '{FRAUD}' AND tag_name LIKE 'class.%'
ORDER BY table_name, column_name"""))

table_name,tags
bronze_accounts,"['nb_access_tier=restricted', 'nb_regulation=UK_GDPR', 'data_classification=pii']"
bronze_customers,"['nb_regulation=UK_GDPR', 'nb_access_tier=restricted', 'data_classification=pii']"
bronze_payees,['nb_access_tier=confidential']
bronze_payments,"['nb_access_tier=confidential', 'nb_regulation=PSR_APP_REIMBURSEMENT']"
bronze_scam_claims,"['nb_regulation=PSR_APP_REIMBURSEMENT', 'nb_access_tier=confidential']"
gold_daily_fraud_kpis,"['system.certification_status=certified', 'nb_access_tier=internal']"
gold_mule_watchlist,"['nb_access_tier=confidential', 'nb_sharing=interbank_data_sharing']"
gold_payment_features,"['nb_pii_protection=no_direct_identifiers', 'nb_access_tier=internal', 'nb_ml_use=training_and_scoring']"
silver_accounts,"['data_classification=pii', 'nb_pii_protection=column_masked', 'nb_access_tier=confidential']"
silver_customers,"['nb_regulation=UK_GDPR', 'nb_access_tier=confidential', 'data_classification=pii', 'nb_pii_protection=column_masked']"


table_name,column_name,governed_pii_class
bronze_accounts,account_number,class.bank_number
bronze_accounts,sort_code,class.bank_number
bronze_customers,date_of_birth,class.date_of_birth
bronze_customers,email,class.email_address
bronze_customers,full_name,class.name
bronze_customers,mobile_phone,class.phone_number
bronze_customers,postcode,class.location
silver_accounts,account_number,class.bank_number
silver_accounts,sort_code,class.bank_number
silver_customers,date_of_birth,class.date_of_birth


### Column masks bound to silver tables (defined in the pipeline DDL)

In [7]:
display(spark.sql(f"""
SELECT table_name, column_name, mask_name
FROM {CATALOG}.information_schema.column_masks WHERE table_schema = '{FRAUD}' ORDER BY table_name, column_name"""))

table_name,column_name,mask_name
silver_accounts,account_number,serverless_stable_my49a8_catalog.northbridge_fraud.mask_pii_string
silver_accounts,sort_code,serverless_stable_my49a8_catalog.northbridge_fraud.mask_pii_string
silver_customers,date_of_birth,serverless_stable_my49a8_catalog.northbridge_fraud.mask_pii_date
silver_customers,email,serverless_stable_my49a8_catalog.northbridge_fraud.mask_pii_string
silver_customers,full_name,serverless_stable_my49a8_catalog.northbridge_fraud.mask_pii_string
silver_customers,mobile_phone,serverless_stable_my49a8_catalog.northbridge_fraud.mask_pii_string
silver_customers,postcode,serverless_stable_my49a8_catalog.northbridge_fraud.mask_pii_string


### Proof: what a non-PII-reader (this job identity) actually sees

In [9]:
display(spark.sql(f"""
SELECT current_user() AS evaluated_as, is_account_group_member('northbridge-pii-readers') AS is_pii_reader,
       customer_id, full_name, date_of_birth, email, mobile_phone, postcode, postcode_district, age_band, vulnerability_flag
FROM {fq('silver_customers')} ORDER BY customer_id LIMIT 5"""))
display(spark.sql(f"SELECT account_id, sort_code, account_number, product FROM {fq('silver_accounts')} ORDER BY account_id LIMIT 3"))

evaluated_as,is_pii_reader,customer_id,full_name,date_of_birth,email,mobile_phone,postcode,postcode_district,age_band,vulnerability_flag
zeeshan.dar@databricks.com,False,C1000000,S**********s,1989-01-01,s***************************k,0**********0,N*****O,NW3,35-49,False
zeeshan.dar@databricks.com,False,C1000001,J*********r,1962-01-01,j**************************k,0**********8,P*****W,PL1,50-64,False
zeeshan.dar@databricks.com,False,C1000002,M***********s,1937-01-01,m****************************k,0**********7,L*****Q,LN1,75+,True
zeeshan.dar@databricks.com,False,C1000003,A**********n,1985-01-01,a***************************k,0**********5,B*****T,B15,35-49,False
zeeshan.dar@databricks.com,False,C1000004,H***********s,1960-01-01,h****************************k,0**********6,Y*****Y,YO1,65-74,False


account_id,sort_code,account_number,product
A30000000,3******4,2******1,Premier Current Account
A30000001,3******5,5******6,Premier Current Account
A30000002,3******1,3******2,Classic Current Account


### Lineage captured automatically by Unity Catalog

In [11]:
display(spark.sql(f"""
SELECT DISTINCT source_table_full_name, target_table_full_name, entity_type
FROM system.access.table_lineage
WHERE target_table_catalog = '{CATALOG}' AND target_table_schema IN ('{FRAUD}', 'northbridge_ml', 'northbridge_ops')
  AND source_table_full_name IS NOT NULL AND event_date >= current_date() - INTERVAL 7 DAYS
ORDER BY target_table_full_name, source_table_full_name"""))

source_table_full_name,target_table_full_name,entity_type
serverless_stable_my49a8_catalog.northbridge_fraud.gold_payment_features,serverless_stable_my49a8_catalog.northbridge_fraud.gold_daily_fraud_kpis,PIPELINE
serverless_stable_my49a8_catalog.northbridge_fraud.gold_payment_features,serverless_stable_my49a8_catalog.northbridge_fraud.gold_mule_watchlist,PIPELINE
serverless_stable_my49a8_catalog.northbridge_fraud.silver_accounts,serverless_stable_my49a8_catalog.northbridge_fraud.gold_payment_features,PIPELINE
serverless_stable_my49a8_catalog.northbridge_fraud.silver_customers,serverless_stable_my49a8_catalog.northbridge_fraud.gold_payment_features,PIPELINE
serverless_stable_my49a8_catalog.northbridge_fraud.silver_payees,serverless_stable_my49a8_catalog.northbridge_fraud.gold_payment_features,PIPELINE
serverless_stable_my49a8_catalog.northbridge_fraud.silver_payments,serverless_stable_my49a8_catalog.northbridge_fraud.gold_payment_features,PIPELINE
serverless_stable_my49a8_catalog.northbridge_fraud.silver_scam_claims,serverless_stable_my49a8_catalog.northbridge_fraud.gold_payment_features,PIPELINE
serverless_stable_my49a8_catalog.northbridge_fraud.bronze_accounts,serverless_stable_my49a8_catalog.northbridge_fraud.silver_accounts,PIPELINE
serverless_stable_my49a8_catalog.northbridge_fraud.bronze_customers,serverless_stable_my49a8_catalog.northbridge_fraud.silver_customers,PIPELINE
serverless_stable_my49a8_catalog.northbridge_fraud.bronze_payees,serverless_stable_my49a8_catalog.northbridge_fraud.silver_payees,PIPELINE


### Effective grants on the Northbridge schemas

In [13]:
display(spark.sql(f"""
SELECT grantee, schema_name, privilege_type
FROM {CATALOG}.information_schema.schema_privileges
WHERE schema_name LIKE 'northbridge%' ORDER BY schema_name, grantee, privilege_type"""))

grantee,schema_name,privilege_type
account users,northbridge_fraud,SELECT
account users,northbridge_fraud,USE_SCHEMA
aec5be3d-de3c-404c-8e60-feed0f265fd3,northbridge_fraud,ALL_PRIVILEGES
aec5be3d-de3c-404c-8e60-feed0f265fd3,northbridge_fraud,MANAGE
fabee2fc-7b58-4ba9-9885-75ea672ff64c,northbridge_fraud,SELECT
fabee2fc-7b58-4ba9-9885-75ea672ff64c,northbridge_fraud,USE_SCHEMA
zeeshan.dar@databricks.com,northbridge_fraud,ALL_PRIVILEGES
zeeshan.dar@databricks.com,northbridge_fraud,MANAGE
account users,northbridge_ml,EXECUTE
account users,northbridge_ml,SELECT
